**RAG Application**

In [ ]:
# MODEL
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,
    max_tokens=300
)

# DOC PROCESSING
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def doc_processing(file_path):
    loader = PyPDFLoader(file_path)
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    return splitter.split_documents(docs)


# EMBEDDINGS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()


# VECTOR STORE
from langchain_community.vectorstores import FAISS

vector_store = None  # Will initialize after upload


# FASTAPI
from fastapi import FastAPI, UploadFile, File

app = FastAPI(title="Summary App")


# UPLOAD API
@app.post("/upload_doc")
async def upload_doc(file: UploadFile = File(...)):
    global vector_store

    file_path = f"./{file.filename}"
    with open(file_path, "wb") as f:
        f.write(await file.read())

    docs = doc_processing(file_path)

    # Initialize FAISS with documents
    vector_store = FAISS.from_documents(docs, embeddings)

    return {"filename": file.filename, "status": "uploaded"}


# PROMPT TEMPLATE
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant.

Answer the question based only on the context below.

Context:
{context}

Question:
{query}

Rules:
1. Answer must be between 50 and 100 words.
2. Do not use abusive language.
3. If the user uses abusive words, respond with:
   "I dont Know contact HR Teams"
"""
)


# QUERY API
@app.get("/query")
async def query_api(query: str):
    if vector_store is None:
        return {"error": "No document uploaded yet"}

    docs = vector_store.similarity_search(query)
    context = "\n".join([doc.page_content for doc in docs])

    formatted_prompt = prompt.format(
        context=context,
        query=query
    )

    response = model.invoke(formatted_prompt)

    return {"answer": response.content}